## Bronze Layer

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

In [0]:
spark= SparkSession.builder\
                   .appName("Retail Fashion Data")\
                   .getOrCreate()
                   

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retailfashiondata.Silver;

CREATE SCHEMA IF NOT EXISTS retailfashiondata.Gold;

In [0]:

from pyspark.sql.functions import current_timestamp

df_cust = (spark.readStream
           .format("cloudFiles")
           .option("cloudFiles.format", "csv")
           .option("pathGlobFilter", "customer*")  # Corrected option key
           .option("inferSchema", "true")
           .option("header", "true")
           .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
           .option("cloudFiles.schemaLocation", "/Volumes/retailfashiondata/csv_data_raw/schema_location/cutomer_data")
           .option("cloudFiles.backfillInterval","7 days")
           .load("/Volumes/retailfashiondata/csv_data_raw/retailfashiondata-csv/")) 

df_cust= df_cust.withColumn("ingestion_time",current_timestamp())

## Handling late files 

df_cust= df_cust.withWatermark("ingestion_time","7 days")


## Write

# The stream stops automatically because trigger(availableNow=True) processes all available data and then terminates.
(df_cust.writeStream.format("delta")
               .option("checkpointLocation", "/Volumes/retailfashiondata/csv_data_raw/checkpoint_location/customer_data")
               .outputMode("append")
               .option("mergeSchema", "true")
               .trigger(availableNow=True)
               .toTable("retailfashiondata.bronze.customer_data")
               )

In [0]:

%sql
--Drop table retailfashiondata.bronze.product_data;
--select * from retailfashiondata.bronze.customer_data;

In [0]:
spark.table("retailfashiondata.bronze.customer_Table").count()

In [0]:
df_cust.printSchema()

In [0]:

df_prod = (spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format","csv")
              .option("pathGlobFilter","product*")
              .option("header","true")
              .option("inferSchema","true")
              .option("coludFiles.schemaEvolutionMode","addNewColumns")
              .option("cloudFiles.schemaLocation","/Volumes/retailfashiondata/csv_data_raw/schema_location/product_data")
              .option("cloudFiles.backfillInterval","7 days")
              .load("/Volumes/retailfashiondata/csv_data_raw/retailfashiondata-csv/")
              )

df_prod = df_prod .withColumn("ingestion_time",current_timestamp())

## handling late arrival file
df_prod= df_prod.withWatermark("ingestion_time","7 days")

(df_prod.writeStream
           .format("delta")
           .outputMode("append")
           .option("mergeSchema","true")
           .option("checkpointLocation","/Volumes/retailfashiondata/csv_data_raw/checkpoint_location/product_data")
           .trigger(availableNow=True)
           .toTable("retailfashiondata.bronze.product_data"))

In [0]:
%sql
Select * from retailfashiondata.bronze.product_data;

In [0]:

## read_data

df_sales= (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("pathGlobFilter","sales*")
    .option("inferSchema","true")
    .option("header","true")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .option("cloudFiles.schemaLocation","/Volumes/retailfashiondata/csv_data_raw/schema_location/sales_data")
    .option("cloudFiles.backfillInterval","7 days")
    .load("/Volumes/retailfashiondata/csv_data_raw/retailfashiondata-csv/")
)

df_sales= df_sales.withColumn("ingestion_time",current_timestamp())

## handling late arrival file:
df_sales= df_sales.withWatermark("ingestion_time","7 days")
## write

(df_sales.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation","/Volumes/retailfashiondata/csv_data_raw/checkpoint_location/sales_data")
        .option("mergeSchema","true")
        .trigger(availableNow=True)
        .toTable("retailfashiondata.bronze.sales_data")
)

In [0]:
%sql
select * from retailfashiondata.bronze.sales_data;

In [0]:
##read

df_store = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("pathGlobFilter","store*")
    .option("header","true")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .option("cloudFiles.schemaLocation","/Volumes/retailfashiondata/csv_data_raw/schema_location/store_data")
    .option("cloudFiles.backfillInterval","7 days")
    .load("/Volumes/retailfashiondata/csv_data_raw/retailfashiondata-csv/")
)

df_store= df_store.withColumn("ingestion_time",current_timestamp())

# handling late arrival files :
df_store= df_store.withWatermark("ingestion_time","7 days")

## write

( df_store.writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation","/Volumes/retailfashiondata/csv_data_raw/checkpoint_location/store_data")
         .option("mergeSchema","true")
         .trigger(availableNow=True)
         .toTable("retailfashiondata.bronze.store_data")


)

In [0]:
%sql
select * from retailfashiondata.bronze.store_data;